In [ ]:
import os

os.chdir("../")

In [ ]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataSplitConfig:
    root_dir: Path
    data_path: Path

In [8]:
from ares.constants import *
from ares.utils.common import read_yaml, create_directories


In [10]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_split_config(self) -> DataSplitConfig:
        config = self.config.data_split

        create_directories([config.root_dir])

        data_validation_config = DataSplitConfig(
            root_dir=config.root_dir, data_path=config.data_path
        )
        return data_validation_config

In [ ]:
import os
from ares import logger
from sklearn.model_selection import train_test_split
import pandas as pd

In [ ]:
class DataSplit:
    def __init__(self, config: DataSplitConfig):
        self.config = config

    def split(self):
        data = pd.read_csv(self.config.data_path)

        MIN_LISTINGS = 50
        locality_counts = data["locality"].value_counts()
        rare_localitys = locality_counts[locality_counts < MIN_LISTINGS].index

        data["locality_grouped"] = data["locality"].where(
            ~data["locality"].isin(rare_localitys), other="OTHER"
        )

        train, eval = train_test_split(
            data, test_size=0.2, random_state=2025, stratify=data["locality_grouped"]
        )

        train.to_csv(os.path.join(self.config.root_dir, "train.csv"), index=False)
        eval.to_csv(os.path.join(self.config.root_dir, "eval.csv"), index=False)

        logger.info("Splited data into training and evaluation sets")
        logger.info(train.shape)
        logger.info(eval.shape)


In [13]:
try:
    config = ConfigurationManager()
    data_split_config = config.get_data_split_config()
    data_split = DataSplit(config=data_split_config)
    data_split.split()
except Exception as e:
    raise e

[2026-01-11 19:36:39,792: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-01-11 19:36:39,801: INFO: common: yaml file: params.yaml loaded successfully]
[2026-01-11 19:36:39,811: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-01-11 19:36:39,817: INFO: common: Created directory at: artifacts]
[2026-01-11 19:36:39,826: INFO: common: Created directory at: artifacts/pipeline/data_split]
[2026-01-11 19:36:40,435: INFO: 4246999211: Splited data into training and evaluation sets]
[2026-01-11 19:36:40,438: INFO: 4246999211: (12538, 30)]
[2026-01-11 19:36:40,442: INFO: 4246999211: (3135, 30)]
